# Baseline — Cadastros suspeitos de clientes

*Dupla:* Emilly e Cleber

## Objetivo
Treinar um primeiro modelo Random Forest para identificar cadastros
sinalizados pelas regras selecionadas da Silver.

## Dados e preparação
A preparação é executada pelo notebook FE_clientes_cadastros_suspeitos:
- Leitura das tabelas Silver de clientes e endereços.
- Criação dos atributos de e-mail, hash, idade e endereços.
- Consolidação de cópias idênticas.
- Criação do rótulo experimental rotulo_revisao_v2.
- Separação temporal entre treino e avaliação.
- Codificação dos domínios usando somente o treino.

## Avaliação
Os lotes a partir de 16/10/2026 ficam reservados para avaliação.

O rótulo indica sinalização pelas regras, não confirmação de fraude.
As métricas medirão concordância com essas regras.

In [0]:
%run ./FE_clientes_cadastros_suspeitos

In [0]:
#============================================================
# CONFERÊNCIA DAS ENTRADAS DO BASELINE
# ============================================================

variaveis_necessarias = [
    "df_treino_codificado",
    "df_avaliacao_codificada",
    "features_finais",
    "alvo_modelo",
]

faltantes = [
    nome
    for nome in variaveis_necessarias
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        f"Variáveis ausentes: {faltantes}. "
        "Execute o notebook de engenharia de features."
    )

features_baseline = list(features_finais)

if not features_baseline:
    raise ValueError("A lista de atributos está vazia.")

if len(features_baseline) != len(set(features_baseline)):
    raise ValueError("Existem atributos repetidos na lista.")

for nome, df in [
    ("TREINO", df_treino_codificado),
    ("AVALIAÇÃO", df_avaliacao_codificada),
]:
    colunas_ausentes = [
        coluna
        for coluna in features_baseline + [alvo_modelo]
        if coluna not in df.columns
    ]

    if colunas_ausentes:
        raise ValueError(
            f"Colunas ausentes em {nome}: {colunas_ausentes}"
        )

    total = df.count()

    if total == 0:
        raise ValueError(f"A base {nome} está vazia.")

    print(f"{nome}: {total} registros")

print("\n===== ATRIBUTOS DO BASELINE =====")
for coluna in features_baseline:
    print(f"- {coluna}")

print("\nQuantidade de atributos:", len(features_baseline))
print("Alvo:", alvo_modelo)
print("\nEntradas disponíveis para o treinamento.")
print("Nenhum modelo foi treinado por esta célula.")

In [0]:
# ============================================================
# TREINAMENTO DO BASELINE RANDOM FOREST
# ============================================================

import numpy as np
from importlib.metadata import version
from sklearn.ensemble import RandomForestClassifier

# 1. Verificar as variáveis necessárias
variaveis_necessarias = [
    "df_treino_codificado",
    "features_finais",
    "alvo_modelo",
]

faltantes = [
    nome
    for nome in variaveis_necessarias
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        f"Variáveis ausentes: {faltantes}. "
        "Execute as células 2 e 3."
    )

versao_sklearn = version("scikit-learn")
features_baseline = list(features_finais)

# 2. Validar a lista de atributos
if not features_baseline:
    raise ValueError("A lista de atributos está vazia.")

if len(features_baseline) != len(set(features_baseline)):
    raise ValueError("Existem atributos repetidos na lista.")

colunas_proibidas = {
    "id_cliente",
    "bronze_ingested_at",
    "bronze_source_file",
    "dominio_email",
    "regra_04_falhou",
    "regra_05_falhou",
    "regra_09_falhou",
    "motivo_hash",
    "motivo_data_silver",
    "motivo_dominio",
    alvo_modelo,
}

indevidas = sorted(
    set(features_baseline).intersection(colunas_proibidas)
)

if indevidas:
    raise ValueError(
        f"Colunas indevidas entre os preditores: {indevidas}"
    )

colunas_necessarias = (
    ["id_cliente", "bronze_ingested_at"]
    + features_baseline
    + [alvo_modelo]
)

colunas_ausentes = [
    coluna
    for coluna in colunas_necessarias
    if coluna not in df_treino_codificado.columns
]

if colunas_ausentes:
    raise ValueError(
        f"Colunas ausentes no treino: {colunas_ausentes}"
    )

# 3. Converter somente a base de treino para pandas
# O limite evita uma coleta excessiva para a memória do driver.
limite_registros = 100_000

treino_pd = (
    df_treino_codificado
    .orderBy("id_cliente", "bronze_ingested_at")
    .select(*features_baseline, alvo_modelo)
    .limit(limite_registros + 1)
    .toPandas()
)

if len(treino_pd) > limite_registros:
    raise ValueError(
        "O treino ultrapassa o limite de coleta para pandas."
    )

if treino_pd.empty:
    raise ValueError("A base de treino está vazia.")

X_treino = treino_pd[features_baseline].astype("float64")
y_treino = treino_pd[alvo_modelo]

# 4. Validar os valores antes do treinamento
if not np.isfinite(X_treino.to_numpy()).all():
    raise ValueError(
        "Existem valores nulos, NaN ou infinitos nos atributos."
    )

if y_treino.isna().any() or not y_treino.isin([0, 1]).all():
    raise ValueError(
        "O alvo deve conter somente 0 e 1, sem valores nulos."
    )

y_treino = y_treino.astype("int64")

if set(y_treino.unique()) != {0, 1}:
    raise ValueError(
        "A base de treino precisa conter as duas classes."
    )

# 5. Definir os parâmetros do primeiro modelo
parametros_rf = {
    "n_estimators": 100,
    "max_depth": 5,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": 1,
}

# 6. Treinar usando somente os dados de treino
modelo_rf = RandomForestClassifier(**parametros_rf)
modelo_rf.fit(X_treino, y_treino)

# 7. Exibir o resumo do treinamento
print("===== RANDOM FOREST TREINADO =====")
print("Versão do scikit-learn:", versao_sklearn)
print("Registros de treino:", len(X_treino))
print("Quantidade de atributos:", len(features_baseline))
print("Alvo:", alvo_modelo)

print("\n===== CLASSES NO TREINO =====")
for classe, quantidade in y_treino.value_counts().sort_index().items():
    print(f"Classe {classe}: {quantidade} registros")

print("\n===== PARÂMETROS =====")
for parametro, valor in parametros_rf.items():
    print(f"{parametro}: {valor}")

print("\n===== ATRIBUTOS UTILIZADOS =====")
for coluna in features_baseline:
    print(f"- {coluna}")

print("\nOs dados de avaliação não foram usados no treinamento.")
print("O modelo aprende a reproduzir o rótulo experimental.")
print("Esse rótulo não representa confirmação de fraude.")
print("Nenhum dado da Silver foi alterado.")
print("O modelo não foi salvo explicitamente por esta célula.")

In [0]:
# ============================================================
# AVALIAÇÃO DO BASELINE
# Avaliação nos lotes reservados, sem novo treinamento.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
)

# 1. Validar as dependências
necessarias = [
    "modelo_rf",
    "df_avaliacao_codificada",
    "features_baseline",
    "alvo_modelo",
]

faltantes = [
    nome for nome in necessarias
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        f"Variáveis ausentes: {faltantes}. "
        "Execute as células anteriores."
    )

if list(modelo_rf.feature_names_in_) != list(features_baseline):
    raise ValueError(
        "A lista ou a ordem dos atributos difere do treinamento."
    )

# Limiar definido antes de consultar os resultados.
limiar_baseline = 0.5

# 2. Preparar a avaliação na mesma ordem de atributos do treino
limite_avaliacao = 100_000

avaliacao_pd = (
    df_avaliacao_codificada
    .orderBy("id_cliente", "bronze_ingested_at")
    .select(
        "id_cliente",
        "bronze_ingested_at",
        *features_baseline,
        alvo_modelo,
    )
    .limit(limite_avaliacao + 1)
    .toPandas()
)

if len(avaliacao_pd) > limite_avaliacao:
    raise ValueError("Avaliação acima do limite de coleta para pandas.")

if avaliacao_pd.empty:
    raise ValueError("A base de avaliação está vazia.")

X_avaliacao = avaliacao_pd[features_baseline].astype("float64")
y_avaliacao = avaliacao_pd[alvo_modelo]

if not np.isfinite(X_avaliacao.to_numpy()).all():
    raise ValueError(
        "Existem valores nulos, NaN ou infinitos nos atributos."
    )

if (
    y_avaliacao.isna().any()
    or not y_avaliacao.isin([0, 1]).all()
):
    raise ValueError("O alvo deve conter somente 0 e 1, sem nulos.")

y_avaliacao = y_avaliacao.astype("int64")

# 3. Gerar os escores e as classificações
indice_classe_1 = list(modelo_rf.classes_).index(1)

escores_avaliacao = modelo_rf.predict_proba(
    X_avaliacao
)[:, indice_classe_1]

predicoes_avaliacao = (
    escores_avaliacao >= limiar_baseline
).astype("int64")

# Referência simples: classificar todos como não sinalizados.
predicoes_referencia = np.zeros(len(y_avaliacao), dtype="int64")

# 4. Comparar as métricas da classe 1
def calcular_metricas(nome, predicoes):
    return {
        "modelo": nome,
        "acuracia": accuracy_score(y_avaliacao, predicoes),
        "precisao_classe_1": precision_score(
            y_avaliacao, predicoes, zero_division=0
        ),
        "recall_classe_1": recall_score(
            y_avaliacao, predicoes, zero_division=0
        ),
        "f1_classe_1": f1_score(
            y_avaliacao, predicoes, zero_division=0
        ),
    }

metricas_baseline = pd.DataFrame([
    calcular_metricas("Random Forest", predicoes_avaliacao),
    calcular_metricas("Sempre classe 0", predicoes_referencia),
])

print("===== AVALIAÇÃO DO BASELINE =====")
print("Registros:", len(y_avaliacao))
print("Limiar:", limiar_baseline)

print("\n===== CLASSES REAIS =====")
for classe, quantidade in y_avaliacao.value_counts().sort_index().items():
    print(f"Classe {classe}: {quantidade} registros")

print("\n===== MÉTRICAS — ESCALA DE 0 A 1 =====")
display(metricas_baseline.round(4))

# 5. Exibir a matriz de confusão
matriz = confusion_matrix(
    y_avaliacao,
    predicoes_avaliacao,
    labels=[0, 1],
)

matriz_confusao_baseline = pd.DataFrame(
    matriz,
    index=["Real 0", "Real 1"],
    columns=["Previsto 0", "Previsto 1"],
)

print("\n===== MATRIZ DE CONFUSÃO =====")
display(matriz_confusao_baseline.reset_index(names="classe_real"))

tn, fp, fn, tp = matriz.ravel()

print("Verdadeiros negativos:", tn)
print("Falsos positivos:", fp)
print("Falsos negativos:", fn)
print("Verdadeiros positivos:", tp)

# 6. Métricas de ordenação dos escores
if y_avaliacao.nunique() == 2:
    print("\n===== MÉTRICAS DOS ESCORES =====")
    print(
        "Average Precision (AP):",
        round(
            average_precision_score(y_avaliacao, escores_avaliacao),
            4,
        ),
    )
    print(
        "ROC AUC:",
        round(
            roc_auc_score(y_avaliacao, escores_avaliacao),
            4,
        ),
    )
else:
    print("\nAvaliação com uma única classe: AP e ROC AUC não exibidas.")

# 7. Manter os resultados para análise, sem gravar dados
resultados_avaliacao = avaliacao_pd[
    ["id_cliente", "bronze_ingested_at", alvo_modelo]
].copy()

resultados_avaliacao["escore_classe_1"] = escores_avaliacao
resultados_avaliacao["predicao"] = predicoes_avaliacao

print("\nAs métricas medem concordância com o rótulo experimental.")
print("O escore não representa probabilidade comprovada de fraude.")
print("Nenhum parâmetro foi ajustado nesta avaliação.")
print("Nenhum modelo foi retreinado e nenhum dado foi gravado.")

## Conclusão do baseline

O Random Forest foi treinado com 1.063 registros e 17 atributos.
A avaliação temporal utilizou 72 registros de lotes posteriores,
sem compartilhamento de clientes ou arquivos com o treino.

Com limiar de 0,5, o modelo classificou corretamente os 67 registros
da classe 0 e os 5 registros da classe 1, obtendo acurácia,
precisão, recall e F1 de 100% nesta avaliação.

A referência que sempre prevê classe 0 alcançou acurácia de 93,06%,
mas não identificou nenhum registro positivo.

### Limitações
- O rótulo é experimental, derivado de regras da Silver.
- As métricas representam concordância com essas regras,
  não confirmação de fraude.
- A avaliação contém apenas 5 positivos.
- Todos os registros de avaliação possuem endereço disponível.
- O resultado não garante o mesmo desempenho em novos lotes.

### Próximo passo
Preservar esta configuração como baseline e avaliar novos lotes
ou rótulos revisados, sem ajustar parâmetros com base neste teste.

In [0]:
# ============================================================
# EXPERIMENTOS COM VALIDAÇÃO TEMPORAL
# Usa somente a base de treino.
# Não altera modelo_rf nem utiliza a avaliação reservada.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# 1. Conferir as dependências
necessarias = [
    "df_treino_preparado",
    "alvo_modelo",
    "parametros_rf",
]

faltantes = [
    nome for nome in necessarias
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        f"Variáveis ausentes: {faltantes}. "
        "Execute as células anteriores."
    )

atributos_numericos = [
    "tamanho_usuario_email",
    "qtd_digitos_email",
    "tamanho_hash",
    "idade_cadastro_dias",
    "qtd_enderecos_disponiveis",
    "tem_endereco_disponivel",
    "proporcao_logradouros_ausentes",
]

atributos_endereco = [
    "qtd_enderecos_disponiveis",
    "tem_endereco_disponivel",
    "proporcao_logradouros_ausentes",
]

# 2. Coletar somente o treino, com limite de segurança
limite = 100_000

dados_experimentos = (
    df_treino_preparado
    .orderBy("bronze_ingested_at", "id_cliente")
    .select(
        "id_cliente",
        "bronze_ingested_at",
        "dominio_email",
        *atributos_numericos,
        alvo_modelo,
    )
    .limit(limite + 1)
    .toPandas()
)

if dados_experimentos.empty or len(dados_experimentos) > limite:
    raise ValueError("Base vazia ou acima do limite de coleta.")

if dados_experimentos["id_cliente"].duplicated().any():
    raise ValueError("Existem clientes repetidos na base de treino.")

if dados_experimentos[
    ["bronze_ingested_at", "dominio_email", alvo_modelo]
].isna().any().any():
    raise ValueError("Existem datas, domínios ou rótulos nulos.")

if not dados_experimentos[alvo_modelo].isin([0, 1]).all():
    raise ValueError("O rótulo deve conter somente 0 e 1.")

if not np.isfinite(
    dados_experimentos[atributos_numericos]
    .astype("float64")
    .to_numpy()
).all():
    raise ValueError("Existem atributos numéricos nulos ou infinitos.")

dados_experimentos["dia_ingestao"] = pd.to_datetime(
    dados_experimentos["bronze_ingested_at"]
).dt.date

dias = sorted(dados_experimentos["dia_ingestao"].unique())

if len(dias) < 2:
    raise ValueError("São necessários pelo menos dois dias de treino.")

# 3. Definir experimentos: uma alteração por versão
experimentos = [
    {
        "nome": "original",
        "retirar": [],
        "parametros": {},
    },
    {
        "nome": "sem_idade",
        "retirar": ["idade_cadastro_dias"],
        "parametros": {},
    },
    {
        "nome": "sem_enderecos",
        "retirar": atributos_endereco,
        "parametros": {},
    },
    {
        "nome": "sem_dominio",
        "retirar": ["dominio_email"],
        "parametros": {},
    },
    {
        "nome": "sem_tamanho_hash",
        "retirar": ["tamanho_hash"],
        "parametros": {},
    },
    {
        "nome": "profundidade_3",
        "retirar": [],
        "parametros": {"max_depth": 3},
    },
]

# 4. Preparar os atributos de cada divisão temporal
def preparar_divisao(base_treino, base_validacao, retirar):
    numericas = [
        coluna for coluna in atributos_numericos
        if coluna not in retirar
    ]

    X_t = base_treino[numericas].astype("float64").copy()
    X_v = base_validacao[numericas].astype("float64").copy()

    # O vocabulário de domínio é aprendido apenas no treino
    # desta divisão temporal.
    if "dominio_email" not in retirar:
        dominios = sorted(base_treino["dominio_email"].unique())

        for numero, dominio in enumerate(dominios, start=1):
            coluna = f"dominio_{numero:02d}"

            X_t[coluna] = (
                base_treino["dominio_email"] == dominio
            ).astype("float64")

            X_v[coluna] = (
                base_validacao["dominio_email"] == dominio
            ).astype("float64")

        X_t["dominio_desconhecido"] = (
            ~base_treino["dominio_email"].isin(dominios)
        ).astype("float64")

        X_v["dominio_desconhecido"] = (
            ~base_validacao["dominio_email"].isin(dominios)
        ).astype("float64")

    return X_t, X_v

# 5. Treinar e validar em períodos sucessivos
resultados = []
limiar_experimentos = 0.5

for dia_validacao in dias[1:]:
    treino_periodo = dados_experimentos.loc[
        dados_experimentos["dia_ingestao"] < dia_validacao
    ]

    validacao_periodo = dados_experimentos.loc[
        dados_experimentos["dia_ingestao"] == dia_validacao
    ]

    y_t = treino_periodo[alvo_modelo].astype("int64")
    y_v = validacao_periodo[alvo_modelo].astype("int64")

    if y_t.nunique() < 2:
        print(
            f"Divisão {dia_validacao} não executada: "
            "treino sem as duas classes."
        )
        continue

    for experimento in experimentos:
        X_t, X_v = preparar_divisao(
            treino_periodo,
            validacao_periodo,
            experimento["retirar"],
        )

        parametros = dict(parametros_rf)
        parametros.update(experimento["parametros"])

        modelo_experimento = RandomForestClassifier(**parametros)
        modelo_experimento.fit(X_t, y_t)

        indice_positivo = list(
            modelo_experimento.classes_
        ).index(1)

        escores = modelo_experimento.predict_proba(
            X_v
        )[:, indice_positivo]

        predicoes = (
            escores >= limiar_experimentos
        ).astype("int64")

        tem_positivos = bool((y_v == 1).any())

        resultados.append({
            "experimento": experimento["nome"],
            "dia_validacao": str(dia_validacao),
            "registros_treino": len(y_t),
            "registros_validacao": len(y_v),
            "positivos_validacao": int((y_v == 1).sum()),
            "atributos": X_t.shape[1],
            "acuracia": accuracy_score(y_v, predicoes),
            "precisao": precision_score(
                y_v, predicoes, zero_division=0
            ),
            # Sem positivos reais, não há como medir recall.
            "recall": (
                recall_score(y_v, predicoes, zero_division=0)
                if tem_positivos else np.nan
            ),
            "f1": (
                f1_score(y_v, predicoes, zero_division=0)
                if tem_positivos else np.nan
            ),
            "falsos_positivos": int(
                ((predicoes == 1) & (y_v.to_numpy() == 0)).sum()
            ),
            "falsos_negativos": int(
                ((predicoes == 0) & (y_v.to_numpy() == 1)).sum()
            ),
        })

if not resultados:
    raise ValueError("Nenhuma divisão temporal pôde ser executada.")

# 6. Resumir as métricas sem escolher o modelo automaticamente
resultados_experimentos = pd.DataFrame(resultados)

resumo_experimentos = (
    resultados_experimentos
    .groupby("experimento", as_index=False)
    .agg(
        divisoes=("dia_validacao", "count"),
        divisoes_com_positivos=("recall", "count"),
        acuracia_media=("acuracia", "mean"),
        precisao_media=("precisao", "mean"),
        recall_medio=("recall", "mean"),
        f1_medio=("f1", "mean"),
        f1_minimo=("f1", "min"),
        falsos_positivos=("falsos_positivos", "sum"),
        falsos_negativos=("falsos_negativos", "sum"),
    )
    .sort_values(
        ["f1_medio", "recall_medio"],
        ascending=False,
    )
)

print("===== COMPARAÇÃO DOS EXPERIMENTOS =====")
print("Métricas na escala de 0 a 1.")
display(resumo_experimentos.round(4))

print("\n===== RESULTADOS POR DIA =====")
display(
    resultados_experimentos
    .sort_values(["dia_validacao", "experimento"])
    .round(4)
)

print("\nAs médias atribuem o mesmo peso a cada divisão temporal.")
print("Recall e F1 nulos indicam validação sem positivos reais.")
print("O baseline original não foi substituído.")
print("Os 72 registros reservados não foram utilizados.")
print("Nenhum dado foi gravado ou alterado na Silver.")

In [0]:
# RESUMO DOS EXPERIMENTOS EM TEXTO

for _, linha in resumo_experimentos.iterrows():
    print(f"\n===== {linha['experimento']} =====")
    print(f"Acurácia média: {linha['acuracia_media']:.2%}")
    print(f"Precisão média: {linha['precisao_media']:.2%}")
    print(f"Recall médio:   {linha['recall_medio']:.2%}")
    print(f"F1 médio:       {linha['f1_medio']:.2%}")
    print(f"F1 mínimo:      {linha['f1_minimo']:.2%}")
    print(f"Falsos positivos: {int(linha['falsos_positivos'])}")
    print(f"Falsos negativos: {int(linha['falsos_negativos'])}")

Foram comparadas seis configurações de Random Forest em três divisões temporais dentro da base de treino. A versão sem idade do cadastro apresentou o melhor desempenho, elevando o F1 médio de 96,61% para 100% e eliminando os três erros observados na configuração original. Essa versão foi selecionada como candidata à evolução do baseline. Os 72 registros de teste anteriores não participaram da seleção. A melhora se refere à validação interna e ainda precisa ser confirmada em novos lotes independentes.

In [0]:
# ============================================================
# CANDIDATA V2 SEM IDADE DO CADASTRO
# Escolhida pelas validações temporais dentro do treino.
# Não utiliza os 72 registros de avaliação.
# ============================================================

from sklearn.ensemble import RandomForestClassifier
import numpy as np

necessarias = [
    "X_treino",
    "y_treino",
    "features_baseline",
    "parametros_rf",
    "alvo_modelo",
]

faltantes = [
    nome for nome in necessarias
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        f"Variáveis ausentes: {faltantes}. "
        "Execute a célula de treinamento original."
    )

# 1. Preservar a ordem e retirar somente a idade
if "idade_cadastro_dias" not in features_baseline:
    raise ValueError(
        "A lista original não contém idade_cadastro_dias. "
        "Confira a configuração antes de continuar."
    )

features_v2 = [
    coluna
    for coluna in features_baseline
    if coluna != "idade_cadastro_dias"
]

X_treino_v2 = X_treino.loc[:, features_v2].copy()

if X_treino_v2.empty:
    raise ValueError("A base de treino está vazia.")

if not X_treino_v2.index.equals(y_treino.index):
    raise ValueError("Os atributos e o alvo estão desalinhados.")

if not np.isfinite(X_treino_v2.to_numpy()).all():
    raise ValueError(
        "Existem valores nulos ou infinitos nos atributos."
    )

if y_treino.isna().any() or set(y_treino.unique()) != {0, 1}:
    raise ValueError("O alvo deve conter as duas classes, sem nulos.")

# 2. Manter os parâmetros do experimento sem idade
parametros_rf_v2 = dict(parametros_rf)

# Novo objeto: o modelo_rf original permanece preservado.
modelo_rf_v2 = RandomForestClassifier(**parametros_rf_v2)
modelo_rf_v2.fit(X_treino_v2, y_treino)

# 3. Documentar a versão
print("===== CANDIDATA V2 TREINADA =====")
print("Registros de treino:", len(X_treino_v2))
print("Quantidade de atributos:", len(features_v2))
print("Atributo retirado: idade_cadastro_dias")
print("Alvo:", alvo_modelo)

print("\n===== ATRIBUTOS UTILIZADOS =====")
for coluna in features_v2:
    print(f"- {coluna}")

print("\n===== PARÂMETROS =====")
for parametro, valor in parametros_rf_v2.items():
    print(f"{parametro}: {valor}")

print("\nO modelo original permanece em modelo_rf.")
print("A candidata V2 está em modelo_rf_v2.")
print("Os registros de avaliação não foram utilizados.")
print("Esta célula não calcula novas métricas.")
print("A V2 ainda precisa de avaliação em novos lotes independentes.")
print("Nenhum dado da Silver foi alterado.")
print("O modelo não foi salvo explicitamente por esta célula.")